In [52]:
import torch
from torch.optim.optimizer import Optimizer

In [101]:
inner_prod_param_dict={'Canonical':0.5, 'Euclidean':0.0}

def matrix_square_root(mat_a, mat_a_size, iter_count=100, ridge_epsilon=1e-4):
  """
  Stable iterations for the matrix square root, Nicholas J. Higham
  Page 231, Eq 2.6b
  http://citeseerx.ist.psu.edu/viewdoc/download?doi=10.1.1.6.8799&rep=rep1&type=pdf

  Modified from TensorFlow implementation of https://www.tensorflow.org/api_docs/python/tf/linalg/sqrtm
  """

  def _iter_body(i, mat_y, unused_old_mat_y, mat_z, unused_old_mat_z, err,
                 unused_old_err):
    current_iterate = 0.5 * (3.0 * identity - torch.matmul(mat_z, mat_y))
    current_mat_y = torch.matmul(mat_y, current_iterate)
    current_mat_z = torch.matmul(current_iterate, mat_z)
    # Compute the error in approximation.
    mat_sqrt_a = current_mat_y * torch.sqrt(norm)
    mat_a_approx = torch.matmul(mat_sqrt_a, mat_sqrt_a)
    residual = mat_a - mat_a_approx
    current_err = torch.norm(residual, p=2) / norm
    return i + 1, current_mat_y, mat_y, current_mat_z, mat_z, current_err, err

  identity = torch.eye(mat_a_size, device=mat_a.device, dtype=mat_a.dtype)
  mat_a = mat_a + ridge_epsilon * identity
  norm = torch.norm(mat_a, p=2)
  mat_init_y = mat_a / norm
  mat_init_z = identity
  init_err = norm

  func_input=[0, mat_init_y, mat_init_y, mat_init_z, mat_init_z, init_err, init_err + 1.0]
  for _ in range(iter_count):
    func_input=_iter_body(*func_input)
  return func_input[2] * torch.sqrt(norm), func_input[4] / torch.sqrt(norm)
def matrix_root(A):
  A_root, _ = matrix_square_root(A, A.shape[0], ridge_epsilon=0)
  return A_root

def matrix_root_inv(A, iter_count=100):
  _, A_root_inv = matrix_square_root(A, A.shape[0], ridge_epsilon=0, iter_count=iter_count)
  return A_root_inv

### Compute matrix root inversion by SVD. Super expensive. For debug only
def mat_root_inv_for_debug(A):
    D, U=torch.symeig(A, eigenvectors=True)
    return U@torch.diag(1/torch.sqrt(D))@U.t()

def cayley(Y, alpha=1.0):
    return torch.linalg.inv(torch.eye(Y.shape[0],device=Y.device, dtype=Y.dtype).add(Y, alpha=-alpha/2))@(torch.eye(Y.shape[0],device=Y.device, dtype=Y.dtype).add(Y, alpha=alpha/2))

def _update_func_Stiefel_Adam(X, Y, V, X_grad, p_Y, p_V, step, square, a, b, lr, beta_1, beta_2, expm_method, inner_iter, epsilon):
    bias_correction_1 = 1 - beta_1 ** step
    bias_correction_2 = 1 - beta_2 ** step
    Xt_Xgrad=torch.matmul(X.t(), X_grad)
    grad_Y=(1-b)/2*(Xt_Xgrad-Xt_Xgrad.t())
    if not square:
        grad_V=-(X@Xt_Xgrad-X_grad)
    # Dynamics phi_2 (will be skipped when n=m)
    if not square:
        p_V.mul_(beta_2).add_(grad_V**2, alpha=1-beta_2)
    # Dynamics phi_1
    Y.mul_(beta_1).add_(grad_Y, alpha=-(1-beta_1))
    p_Y.mul_(beta_2).add_(grad_Y**2, alpha=1-beta_2)
    denominator_Y=torch.sqrt(p_Y/bias_correction_2)+epsilon
    xi=lr/bias_correction_1*Y/denominator_Y
    if expm_method=='Cayley':
        X.copy_(X.matmul(cayley(xi)))
    elif expm_method=='MatrixExp':
        X.copy_(X.matmul(torch.matrix_exp(xi)))
    elif expm_method=='ForwardEuler':
        X.add_(X@xi)
    else:
        raise NotImplementedError()
    # Dynamics phi_3 (will be skipped when n=m)
    if not square:
        V.mul_(beta_1)
        V.add_(V@Y, alpha=-(3*a-2)/2*lr/beta_1 if beta_1!=0 else 0.0)
        V.add_(grad_V, alpha=-(1-beta_1))
        denominator_V=torch.sqrt(p_V/bias_correction_2)+epsilon
        V_tilde=V/denominator_V-X@torch.linalg.inv(X.t()@X)@(X.t()@(V/denominator_V))
        XVTV=X@(V_tilde.t()@V)
        X.add_(V_tilde@(X.t()@X), alpha=lr)
        V.add_(XVTV, alpha=-lr)

    X.copy_(X.matmul(matrix_root_inv(X.t()@X, iter_count=inner_iter)))

class StiefelAdam(Optimizer):
    r""" Implementation of Adam on Stiefel manifold from the paper
    Momentum Stiefel Optimizer, with Applications to Suitably-Orthogonal Attention, and Optimal Transport (https://arxiv.org/abs/2205.14173)
    Purpose:
        Given a function f(X), find the minimum value of f under constraint that X has orthonormal columns. This is the adaptive learning version. Suitable for machine learning problems.
    Args:
        - params: A list of matrices. Containing parameters to optimize. 
        - lr (float, optional): learning rate (default: 0.001)
        - betas (Tuple[float, float], optional): coefficients used for computing running averages of gradient and its square (default: (0.9, 0.999))
        - expm_method (str in ['MatrixExp', 'Cayley', 'ForwardEuler'], optional): method to compute matrix exponential. (default: 'ForwardEuler')
        - inner_prod: (float number less than 1 or string in `['Canonical', 'Euclidean']`, optional): the parameter in the canonical-type metric (defined in Definition 1 in the paper).
        - max_inner_iter: (int, optional): maximum number of iterations when computing matrix root inversion. (default: 100)

    Discussion: 
        - We recommend using the same hyperparameters when the model contains both Euclidean parameters and Stiefel parameters. See Remark 1 in the paper for details.
        - The matrices being optimized should have number of rows >= number of columns. Otherwise, the matrix will be transposed without warning. For tensors with more than 2 dimensions, all the dimensions will be flattened excepted the first dimension to create a matrix.
        - There is no significant difference when further tuning expm_method, inner_prod and max_inner_iter. Default is good enough to use.
        - No special orthonormal initialization for Stiefel matrices is required. Commonly used element-wise random Gaussian matrices will work and our optimizer will automatically project it onto the Stiefel manifold. However, explicit initialization using `torch.nn.init.orthogonal_` is still recommended.
    """
    def __init__(self, params, lr=0.001, betas=(0.9,0.99), epsilon=1e-5, expm_method='ForwardEuler', inner_prod='Canonical', inner_iter=10):
        if lr < 0.0:
            raise ValueError("Invalid learning rate: {}".format(lr))
        beta_1, beta_2=betas
        if beta_1<0 or beta_1>=1 or beta_2<=0 or beta_2>=1 :
            raise ValueError('beta out of range')
        assert expm_method in ['MatrixExp', 'Cayley', 'ForwardEuler'], 'expm_method not correct'
        if isinstance(inner_prod, str):
            assert inner_prod in inner_prod_param_dict.keys(), 'inner_prod not correct'
            inner_prod_param=inner_prod_param_dict[inner_prod]
        else:
            inner_prod_param=float(inner_prod)
            assert inner_prod_param < 1
        # metric parameter in Definition 1 in the paper
        a=inner_prod_param
        b=a/(a-1)

        defaults = dict(lr=lr, betas=betas, epsilon=epsilon, expm_method=expm_method, a=a, b=b, inner_iter=inner_iter)
        super(StiefelAdam, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(StiefelAdam, self).__setstate__(state)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            for X_raw in group['params']:
                if X_raw.grad is None:
                    continue
                # If X has more than 2 dimensions with shape [*, n, m], we will keep each of n-by-m matrices on Stiefel manifold.
                X=X_raw.view(-1,X_raw.shape[-2], X_raw.shape[-1])
                X_grad=X_raw.grad.view(-1,X_raw.shape[-2], X_raw.shape[-1])
                # X should be a tall and thin matrix (n>m). Otherwise, it will be transposed.
                square = False
                if X.shape[-2]<X.shape[-1]:
                    X=X.transpose(-1,-2)
                    X_grad=X_grad.transpose(-1,-2)
                else:
                    if X.shape[-2]==X.shape[-1]:
                        square = True
                        
                # Make the algorithm compatible with SO(n)
                # In that case, n=m, and we no longer need V
                beta_1, beta_2=group['betas']
                epsilon=group['epsilon']
                expm_method=group['expm_method']
                a=group['a']
                b=group['b']
                inner_iter=group['inner_iter']
                
                param_state = self.state[X_raw]

                if 'Y_buffer' not in param_state:
                    Y = param_state['Y_buffer']=torch.zeros(X.shape[0], X.shape[-1],X.shape[-1], device=X.device, dtype=X.dtype)
                if 'V_buffer' not in param_state and not square:
                    V = param_state['V_buffer']=torch.zeros(X.shape[0], X.shape[-2],X.shape[-1], device=X.device, dtype=X.dtype)
                if 'p_Y_buffer' not in param_state:
                    p_Y = param_state['p_Y_buffer']=torch.zeros(X.shape[0], X.shape[-1],X.shape[-1], device=X.device, dtype=X.dtype)
                if 'p_V_buffer' not in param_state and not square:
                    p_V = param_state['p_V_buffer']=torch.zeros(X.shape[0], X.shape[-2],X.shape[-1], device=X.device, dtype=X.dtype)
                if 'step' not in param_state:
                    num_step=param_state['step']=0
                
                param_state['step']+=1
                Y = param_state['Y_buffer']
                p_Y = param_state['p_Y_buffer']
                if not square:
                    V = param_state['V_buffer']
                    p_V = param_state['p_V_buffer']
                step=param_state['step']

                if square:
                    update_func=lambda X, Y, X_grad, p_Y: _update_func_Stiefel_Adam(X, Y, None, X_grad, p_Y, None, step, square, a, b, lr, beta_1, beta_2, expm_method, inner_iter, epsilon)
                    torch.vmap(update_func, out_dims=None)(X, Y, X_grad, p_Y)
                else:
                    update_func=lambda X, Y, V, X_grad, p_Y, p_V: _update_func_Stiefel_Adam(X, Y, V, X_grad, p_Y, p_V, step, square, a, b, lr, beta_1, beta_2, expm_method, inner_iter, epsilon)
                    torch.vmap(update_func, out_dims=None)(X, Y, V, X_grad, p_Y, p_V)
                # Check the structure for tangent bundle. For debug only. Please comment out.
                # assert torch.norm(X.t()@X-torch.eye(m, dtype=X.dtype, device=X.device))<torch.finfo(X.dtype).eps*torch.numel(Y)*10
                # assert torch.norm(Y.t()+Y)<torch.finfo(X.dtype).eps*torch.numel(Y)*10
                # assert torch.norm(X.t()@V)<torch.finfo(X.dtype).eps*torch.numel(Y)*10

                
        return loss

In [113]:
# generate random unitary by using QR decomposition
def generate_random_unitary(n):
    q, r = torch.linalg.qr(torch.randn(n, n, dtype=torch.float64))
    return q

# calculate the distance from identity matrix
def calculate_distance_from_identity(U):
    return torch.norm(U - torch.eye(U.shape[0], dtype=U.dtype, device=U.device))

# generate random unitary by using QR decomposition
U = generate_random_unitary(9) 
U = U.requires_grad_(True) 
print(torch.linalg.det(U))

# make determinant 1 
U.data[:] = U.data / torch.linalg.det(U.data)
print(torch.linalg.det(U))
# U.data[:] = torch.eye(U.shape[0], dtype=U.dtype, device=U.device)

tensor(1.0000, dtype=torch.float64, grad_fn=<LinalgDetBackward0>)
tensor(1.0000, dtype=torch.float64, grad_fn=<LinalgDetBackward0>)


In [114]:
I = torch.eye(U.shape[0], dtype=U.dtype, device=U.device)

calculate_distance_from_identity(I)

tensor(0., dtype=torch.float64)

In [115]:
calculate_distance_from_identity(U)

tensor(4.7976, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)

In [116]:
torch.linalg.det(U)

tensor(1.0000, dtype=torch.float64, grad_fn=<LinalgDetBackward0>)

In [117]:
optimizer = StiefelAdam([U], lr=0.01, betas=(0.9, 0.999), epsilon=1e-5, expm_method='MatrixExp', inner_prod='Euclidean', inner_iter=10)

for _ in range(10000):
    optimizer.zero_grad()
    loss = calculate_distance_from_identity(U)
    loss.backward()
    optimizer.step()
    print(loss)


tensor(4.7976, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.7715, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.7450, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.7183, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.6912, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.6639, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.6362, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.6083, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.5800, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.5515, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.5227, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.4935, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.4641, dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)
tensor(4.4343, dtype=torch.float64, grad_fn=<Linalg